# Fieldwise: Muse Glimmer benchmark on a Kaggle GPU

Runs the **local (Glimmer)** half of the benchmark with Meta's official Muse Glimmer 30B Q4_K_M GGUF, served by llama.cpp with the same launcher and settings as the Fieldwise hub. Only the hardware differs.

**Before you start** (right-hand panel → Session options):
1. Accelerator: **GPU T4 x2** (if only a single 16 GB GPU is offered, it still runs, more slowly).
2. Internet: **On** (needs a phone-verified Kaggle account).
3. Add Input → your **private** dataset made with `python bench/pack_for_kaggle.py`.

Run the cells in order. `setup` takes roughly 20–40 min (compiling llama.cpp + downloading 18 GB). If the session restarts, run `setup` and `serve` again (`/tmp` is wiped, so it rebuilds and re-downloads); the bench cells then resume where they stopped, as long as `/kaggle/working` survived.

When finished, run `pack` and download **glimmer_results.zip** from the Output panel.

In [ ]:
import glob
SCRIPT = sorted(glob.glob('/kaggle/input/**/kaggle/kaggle_glimmer.py', recursive=True))[0]
print(SCRIPT)
!nvidia-smi

In [ ]:
!python {SCRIPT} setup

In [ ]:
!python {SCRIPT} serve

Smoke test: 3 questions and 3 photos. Every confidence should end in **(logprobs)** and there should be no `ERR` lines. If not, stop here and send the output to Claude.

In [ ]:
!python {SCRIPT} smoke

Full runs. Each prints one line per item with its time. Re-run a cell to continue after an interruption; finished items are skipped and failed ones retried.

In [ ]:
!python {SCRIPT} bench qa

In [ ]:
!python {SCRIPT} bench images

In [ ]:
!python {SCRIPT} pack

## Live clip for the demo video

In a fresh session run the **setup** and **serve** cells first, then this one (skip smoke and the benchmark). It sends demo questions and photos through the real app pipeline with Glimmer answering live and prints what the app would show, with real times. The Meta API is never called.

Pass the IDs that `python bench/demo_examples.py` printed on your laptop, so the clip matches the replayed scenes, e.g. `!python {SCRIPT} demo q-m41 q-c22 img-maize-leaf_spot-003 --pii q-c11` (`--pii` adds a made-up name and phone number to show redaction). With no IDs it picks a few itself. Record the screen while it runs. The `pack` cell then includes `live_demo.jsonl`.

In [ ]:
!python {SCRIPT} demo

## The real app with live Glimmer (optional)

After **setup** and **serve**, this starts the Fieldwise app on this GPU machine and prints a temporary public link (a Cloudflare quick tunnel) to open on your phone or laptop. Glimmer answers live; the Meta API is off here, so low-confidence items are queued. Anyone with the link can use the app while it's up: don't share it, and run the stop cell when you're done. Label recordings "Live on a Kaggle T4 GPU".

In [ ]:
!python {SCRIPT} ui

**With live Muse Spark and SAM 3.1** (optional): first store your Meta key in *Add-ons > Secrets* with the label `MODEL_API_KEY` and tick it for this notebook. Never paste the key into a cell or the dataset. The app pauses Spark after 2 content-filter refusals and stops all Meta calls if the key is refused. Calls are billed to your Meta credits.

In [ ]:
!python {SCRIPT} ui --spark

In [ ]:
!python {SCRIPT} stop